# Recalibrated AAE vs. $t_0$ plot

This notebook reads the newest compatible LAHM/ISSW CSV and recreates the
sample plot. It is designed to keep working as rows are added later.

## How to use it

1. Keep the notebook beside the CSV, or keep both in the project folder with
   the CSV in `GPT`.
2. Leave `CSV_FILE_OVERRIDE = None` to select the newest compatible dated CSV.
   To use one exact file, enter its path in `CSV_FILE_OVERRIDE`.
3. Add future samples to the CSV using the same column names. A row is plotted
   when it has both a `t_0 value` and an `AAE value`; missing values may remain
   blank or `NaN`.
4. For a new location or sample family, use the same `Plot group` text for all
   replicate rows. Known sample-name variants such as Baker summit and Rainier
   Muir are standardized automatically.
5. Run all cells. The PNG is written into the selected CSV's folder.

Each plot group receives a stable color and a legend entry. Future groups add
dots only. The three established sample boxes—Rainier dirt, pine pollen, and
fullerene standards—are updated from rows containing a paired $t_0$ and AAE
measurement. No new boxes are created automatically.


In [ ]:
from pathlib import Path
import colorsys
import hashlib
import re

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd


# Optional: set this to a specific CSV path. Leave it as None to automatically
# select the newest compatible CSV beside the notebook or in the GPT folder.
CSV_FILE_OVERRIDE = None

CSV_PATTERNS = (
    "lahm_issw_all_available_samples_location_groups*.csv",
    "lahm_issw_all_available_samples_current_analysis*.csv",
    "lahm_issw_all_available_samples_exact_sample_groups*.csv",
    "lahm_issw_all_samples_recalibrated_grouped*.csv",
)
CSV_SEARCH_DIRECTORIES = (Path("."), Path("GPT"))
OUTPUT_FILE_NAME = "AAE_t0_recalibrated_future_samples.png"
LABEL_POINTS = False

T0_COLUMN = "t_0 value"
AAE_COLUMN = "AAE value"
LAHM_NAME_COLUMN = "LAHM file name"
ISSW_NAME_COLUMN = "ISSW file name"
GROUP_COLUMN = "Plot group"

# These established groups retain the same color. Any new group receives a
# deterministic color based on its name.
PREFERRED_COLORS = {
    "Mount Adams": "#1f77b4",
    "Mount Baker crater": "#ff7f0e",
    "Mount Baker summit": "#8c564b",
    "Mount Cook": "#17becf",
    "North Pole": "#7f7f7f",
    "Huann": "#aec7e8",
    "POI": "#ffbb78",
    "Sea ice": "#9edae5",
    "Saguyanig ship track": "#98df8a",
    "Pine pollen": "#9467bd",
    "Rainier Muir": "#2ca02c",
    "Rainier summit": "#bcbd22",
    "Rainier 2025-07-03 Site 01 surface": "#e377c2",
    "Rainier 2025-07-03 Site 01 subsurface": "#d62728",
    "Fullerene standards": "#4c4c4c",
    "Rainier dirt": "#a65628",
    "Car exhaust": "#17a589",
}

# Only these established boxes are drawn. Adding a new sample group to the CSV
# adds dots and a legend entry, but never creates a new box.
BOX_SPECS = (
    {
        "group": "Rainier dirt",
        "label": "Rainier dirt",
        "fallback_x": (4.9, 6.9),
        "fallback_y": (0.0, 0.4),
    },
    {
        "group": "Pine pollen",
        "label": "pine pollen",
        "fallback_x": (6.3, 6.7),
        "fallback_y": (1.5, 2.1),
    },
    {
        "group": "Fullerene standards",
        "label": "fullerene standards",
        "fallback_x": (5.1, 5.7),
        "fallback_y": (-1.0, 0.0),
    },
)




In [ ]:
def find_newest_csv():
    """Return the newest compatible CSV, favoring dated filenames."""
    if CSV_FILE_OVERRIDE is not None:
        selected = Path(CSV_FILE_OVERRIDE).expanduser()
        if not selected.is_file():
            raise FileNotFoundError(f"CSV_FILE_OVERRIDE does not exist: {selected}")
        return selected

    candidates = {}
    pattern_count = len(CSV_PATTERNS)
    for pattern_index, pattern in enumerate(CSV_PATTERNS):
        pattern_priority = pattern_count - pattern_index
        for directory in CSV_SEARCH_DIRECTORIES:
            for path in directory.glob(pattern):
                if path.is_file():
                    resolved = path.resolve()
                    candidates[resolved] = max(
                        pattern_priority,
                        candidates.get(resolved, 0),
                    )

    if not candidates:
        searched = ", ".join(str(path) for path in CSV_SEARCH_DIRECTORIES)
        patterns = ", ".join(CSV_PATTERNS)
        raise FileNotFoundError(
            "No compatible CSV was found. "
            f"Searched {searched} for: {patterns}. "
            "Place the CSV beside this notebook, keep it in GPT, or set "
            "CSV_FILE_OVERRIDE."
        )

    def sort_key(item):
        path, pattern_priority = item
        dates = re.findall(r"20\d{2}-\d{2}-\d{2}", path.name)
        date_key = max(dates) if dates else "0000-00-00"
        return date_key, pattern_priority, path.stat().st_mtime_ns, path.name

    return max(candidates.items(), key=sort_key)[0]


def normalized_text(value):
    """Normalize separators without changing the scientific data."""
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("_", " ")).strip()


def normalize_plot_group(row):
    """Group known replicate names; preserve any new CSV group name."""
    existing = normalized_text(row.get(GROUP_COLUMN))
    lahm = normalized_text(row.get(LAHM_NAME_COLUMN))
    issw = normalized_text(row.get(ISSW_NAME_COLUMN))
    combined = f"{existing} {lahm} {issw}".strip()
    lower = combined.lower()
    compact = re.sub(r"[^a-z0-9]+", "", lower)

    if "car exhaust" in lower or re.search(r"\bups2021-e\d+", lower):
        return "Car exhaust"
    if "nucstd" in lower or re.search(r"\bnuc\d+\b", lower):
        return "Nucleopore standards"
    if "fullerene" in lower:
        return "Fullerene standards"
    if "pine pollen" in lower or "pp2026" in lower:
        return "Pine pollen"
    if "baker" in lower and "crater" in lower:
        return "Mount Baker crater"
    if "baker" in lower and "summit" in lower:
        return "Mount Baker summit"
    if "mount adams" in lower or "mtadams" in compact:
        return "Mount Adams"
    if "mount cook" in lower or "mtcook" in compact:
        return "Mount Cook"
    if "north pole" in lower or "northpole" in compact:
        return "North Pole"
    if "huann" in lower:
        return "Huann"
    if re.search(r"\bpoi\b", lower):
        return "POI"
    if "sea ice" in lower or "greenland" in lower:
        return "Sea ice"
    if any(term in lower for term in ("saguan", "saguyan", "sanguyan")):
        return "Saguyanig ship track"
    if (
        "rainier dirt" in lower
        or lower.startswith("dirt ")
        or " dirt str" in lower
    ):
        return "Rainier dirt"
    if "rainier" in lower and "muir" in lower:
        return "Rainier Muir"
    if "rainier" in lower and "summit" in lower:
        return "Rainier summit"

    existing_lower = existing.lower()
    if existing_lower.startswith("2025-07-03-01-"):
        suffix = existing_lower.removeprefix("2025-07-03-01-")
        return (
            "Rainier 2025-07-03 Site 01 surface"
            if suffix in {"1a", "1b"}
            else "Rainier 2025-07-03 Site 01 subsurface"
        )
    rainier_site = re.match(
        r"^(?:rainier )?(20\d{2}-\d{2}-\d{2})[- ](?:site )?(\d{2})[- ]",
        existing_lower,
    )
    if rainier_site:
        return f"Rainier {rainier_site.group(1)} Site {rainier_site.group(2)}"

    if existing_lower.startswith("ups2020-d"):
        return "UPS2020-D"
    if existing_lower.startswith("ups2020-"):
        return "UPS2020"
    ups2021 = re.match(r"^ups2021-([ceprs])", existing_lower)
    if ups2021:
        return f"UPS2021-{ups2021.group(1).upper()}"
    if existing_lower.startswith("ups2021-"):
        return "UPS2021"
    if existing_lower.startswith("cwu2021-r"):
        return "CWU2021-R"

    existing_compact = re.sub(r"[\s._-]+", "", existing_lower)
    if existing_compact == "28ml" or (
        re.match(r"^[abc]\d+ml", existing_compact)
        and not existing_compact.startswith(("nuca", "nucb", "nucc"))
    ):
        return "Ink standards"
    if existing_lower.startswith("nucleopore samples") or existing_compact.startswith(
        ("nuca", "nucb", "nucc")
    ):
        return "Nucleopore samples"
    if existing_lower.startswith(("nuc", "nucleopore standards")):
        return "Nucleopore standards"
    if existing_lower.startswith("ch16"):
        return "Chile samples"
    if existing_lower.startswith("lahm test data"):
        return "LAHM test data"

    # A new location or sample family should have the same Plot group text on
    # all of its CSV rows. The notebook preserves that name automatically.
    if existing:
        return existing

    fallback_name = issw or lahm
    if fallback_name:
        return Path(fallback_name).stem.replace("_", " ")
    return "Unresolved sample"


def color_for_group(group):
    """Return a stable color, including for groups added in future CSVs."""
    if group in PREFERRED_COLORS:
        return PREFERRED_COLORS[group]

    digest = hashlib.sha256(group.encode("utf-8")).digest()
    hue = int.from_bytes(digest[:2], "big") / 65535
    saturation = 0.55 + (digest[2] / 255) * 0.20
    value = 0.68 + (digest[3] / 255) * 0.18
    return colorsys.hsv_to_rgb(hue, saturation, value)


def padded_bounds(values, fallback, minimum_padding, padding_fraction=0.08):
    """Return a padded numeric range, or a documented fallback."""
    numeric_values = pd.to_numeric(values, errors="coerce").dropna()
    if numeric_values.empty:
        return fallback, False

    lower = float(numeric_values.min())
    upper = float(numeric_values.max())
    padding = max(minimum_padding, (upper - lower) * padding_fraction)
    return (lower - padding, upper + padding), True


def add_sample_range_box(ax, data, group, label, fallback_x, fallback_y):
    """Draw one established box using rows with paired t_0 and AAE values."""
    group_data = data[data[GROUP_COLUMN] == group]
    paired_data = group_data.dropna(subset=[T0_COLUMN, AAE_COLUMN])
    ignored_incomplete = len(group_data) - len(paired_data)

    x_bounds, x_updated = padded_bounds(
        paired_data[T0_COLUMN],
        fallback_x,
        minimum_padding=0.05,
    )
    y_bounds, y_updated = padded_bounds(
        paired_data[AAE_COLUMN],
        fallback_y,
        minimum_padding=0.05,
    )
    x_min, x_max = x_bounds
    y_min, y_max = y_bounds
    color = color_for_group(group)

    box_label = (
        f"{label}\n(observed range)"
        if x_updated and y_updated
        else f"{label}\n(previous range)"
    )
    ax.add_patch(
        Rectangle(
            (x_min, y_min),
            x_max - x_min,
            y_max - y_min,
            facecolor=color,
            edgecolor=color,
            linewidth=1.4,
            alpha=0.16,
            zorder=1,
        )
    )
    ax.text(
        (x_min + x_max) / 2,
        (y_min + y_max) / 2,
        box_label,
        ha="center",
        va="center",
        fontsize=9,
        bbox={
            "facecolor": "white",
            "edgecolor": "none",
            "alpha": 0.55,
            "pad": 1.5,
        },
        zorder=2,
    )
    print(
        f"{box_label.replace(chr(10), ' ')} box: "
        f"t_0={x_min:.3f} to {x_max:.3f}; "
        f"AAE={y_min:.3f} to {y_max:.3f}; "
        f"paired rows={len(paired_data)}; "
        f"incomplete rows ignored={ignored_incomplete}"
    )




In [ ]:
CSV_FILE = find_newest_csv()
OUTPUT_FILE = CSV_FILE.parent / OUTPUT_FILE_NAME
print(f"Using CSV: {CSV_FILE}")

data = pd.read_csv(
    CSV_FILE,
    na_values=["NaN", "nan", "NAN", ""],
    keep_default_na=True,
)
data.columns = data.columns.str.strip()

required_columns = {
    LAHM_NAME_COLUMN,
    T0_COLUMN,
    ISSW_NAME_COLUMN,
    AAE_COLUMN,
}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(
        "The CSV is missing these required columns: "
        + ", ".join(sorted(missing_columns))
    )

for column in (LAHM_NAME_COLUMN, ISSW_NAME_COLUMN):
    data[column] = data[column].astype("string").str.strip().replace("", pd.NA)

data[T0_COLUMN] = pd.to_numeric(data[T0_COLUMN], errors="coerce")
data[AAE_COLUMN] = pd.to_numeric(data[AAE_COLUMN], errors="coerce")
data["Sample label"] = data[ISSW_NAME_COLUMN].fillna(data[LAHM_NAME_COLUMN])

if GROUP_COLUMN not in data.columns:
    data[GROUP_COLUMN] = pd.NA
data[GROUP_COLUMN] = data.apply(normalize_plot_group, axis=1)

complete = data.dropna(subset=[T0_COLUMN, AAE_COLUMN]).copy()
incomplete = data[data[[T0_COLUMN, AAE_COLUMN]].isna().any(axis=1)].copy()
if complete.empty:
    raise ValueError("No CSV rows contain both a t_0 value and an AAE value.")

print(f"Loaded {len(data)} CSV rows.")
print(
    f"Plotting {len(complete)} complete rows in "
    f"{complete[GROUP_COLUMN].nunique()} groups."
)
print(f"Skipping {len(incomplete)} rows missing t_0 or AAE.")
print("\nPlotted groups:")
print(complete[GROUP_COLUMN].value_counts().sort_index().to_string())

# Retain the original view, but expand when future samples fall outside it.
x_range = complete[T0_COLUMN].max() - complete[T0_COLUMN].min()
y_range = complete[AAE_COLUMN].max() - complete[AAE_COLUMN].min()
x_padding = max(0.2, 0.05 * x_range)
y_padding = max(0.2, 0.05 * y_range)

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(
    min(3.5, complete[T0_COLUMN].min() - x_padding),
    max(7.5, complete[T0_COLUMN].max() + x_padding),
)
ax.set_ylim(
    min(-1.2, complete[AAE_COLUMN].min() - y_padding),
    max(3.2, complete[AAE_COLUMN].max() + y_padding),
)
ax.set_xlabel(r"$t_0$ (s)")
ax.set_ylabel("Absorption Ångström exponent (AAE)")
ax.grid(True, alpha=0.25)
ax.set_axisbelow(True)

# This literature box remains fixed because it is not calculated from the CSV.
ax.add_patch(
    Rectangle(
        (3.95, 1.5),
        2.5,
        1.5,
        facecolor="red",
        edgecolor="red",
        linewidth=1.4,
        alpha=0.16,
        zorder=1,
    )
)
ax.text(
    5.2,
    2.25,
    "igneous rock\n(literature)",
    ha="center",
    va="center",
    fontsize=9,
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.55, "pad": 1.5},
    zorder=2,
)

for box_spec in BOX_SPECS:
    add_sample_range_box(ax=ax, data=data, **box_spec)

# Every complete CSV row becomes a dot. New group names receive stable colors
# and legend entries automatically. No new boxes are created.
for group, group_data in complete.groupby(GROUP_COLUMN, sort=True):
    ax.scatter(
        group_data[T0_COLUMN],
        group_data[AAE_COLUMN],
        marker="o",
        s=58,
        color=color_for_group(group),
        edgecolor="white",
        linewidth=0.6,
        label=group,
        zorder=3,
    )

    if LABEL_POINTS:
        for _, row in group_data.iterrows():
            ax.annotate(
                row["Sample label"],
                (row[T0_COLUMN], row[AAE_COLUMN]),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=7,
            )

legend_columns = 1 if complete[GROUP_COLUMN].nunique() <= 18 else 2
ax.legend(
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    fontsize=8,
    frameon=True,
    borderaxespad=0,
    ncol=legend_columns,
)
fig.tight_layout()
fig.savefig(OUTPUT_FILE, dpi=300, bbox_inches="tight")
print(f"\nSaved plot: {OUTPUT_FILE}")
plt.show()
